# Module 6 — SHACL: Making the Boundary Mechanical

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS (Amazon Web Services)

---

## What this module teaches

The deterministic-vs-probabilistic boundary is the single most important concept in
ATLAS. Every prior module has referenced it. Module 6 makes it **mechanical** —
enforced by code, not by convention.

The enforcement tool is SHACL (Shapes Constraint Language), a W3C standard for
writing validation rules on graph data. A SHACL shape says: "data that looks like
THIS is valid; data that looks like THAT is a violation." When you run the validator,
it produces a report — a machine-readable document that a Model Risk Management (MRM)
reviewer can read in twenty minutes.

By the end of this module you can:

- Write SHACL shapes that enforce the deterministic-vs-probabilistic boundary
- Run the validator and read the report
- Explain to an MRM reviewer what each shape enforces and why they should care
- Deliberately introduce a violation and watch the validator catch it

## Key Terms for This Module

| Term | What It Is |
|------|------------|
| **SHACL (Shapes Constraint Language)** | A W3C standard for defining validation rules on RDF graph data. SHACL shapes describe what valid data looks like; the validator checks whether actual data conforms. Think of it as "unit tests for your graph." |
| **Shape** | A single validation rule in SHACL. A shape targets a class (e.g., all Score nodes) and defines constraints (e.g., must have a confidence value between 0 and 1). |
| **Conformance** | Whether data passes all SHACL shapes without violations. "The graph conforms" means every node satisfies every shape that targets it. |
| **Violation** | A specific instance where data fails a shape. The validation report lists each violation with the node, the shape, and the constraint that failed. |
| **Target class** | The class of nodes a shape applies to. Example: a shape with target class `atlas:Score` checks every Score node in the graph. |
| **Deterministic component** | A system component that, given the same inputs, always produces the same outputs. The path from input to output is fully expressible in code or formal logic. Safe for compliance decisions. |
| **Probabilistic-explainable component** | A non-deterministic component that produces a per-record explanation alongside its output. Acceptable for compliance use only when paired with explanations and version-pinned models. |
| **Probabilistic-opaque component** | A non-deterministic component with no per-record explanation possible. Cannot be a compliance input. Confined to interface roles (NL↔SPARQL, narrative drafting). |
| **pyshacl** | The Python library that executes SHACL validation. Used throughout this workshop. Pinned version in `notebooks/shared/requirements.txt`. |
| **Validation report** | The output of running SHACL shapes against a data graph. Lists all violations (or confirms conformance). In ATLAS, this report is a deliverable for MRM reviewers. |
| **SR 11-7** | Federal Reserve Supervision and Regulation Letter 11-7 — "Supervisory Guidance on Model Risk Management." The US regulatory framework that requires models to be reproducible and subject to independent validation. |
| **OCC 2011-12** | Office of the Comptroller of the Currency Bulletin 2011-12 — companion to SR 11-7. Together they define what MRM reviewers look for. |

## Why SHACL and not just OWL restrictions

OWL (Web Ontology Language) can express constraints — but OWL constraints are for
**reasoning** (inferring new facts), not for **validation** (catching bad data).

SHACL is specifically designed for validation. It produces a **report** — a document
that says "node X violated shape Y because constraint Z was not met." That report
is what an MRM reviewer reads. OWL does not produce reports; it produces inferences.

In ATLAS:
- **OWL** defines the class hierarchy and enables reasoner-derivable inferences
- **SHACL** validates data integrity and enforces the boundary

They are complementary, not alternatives.

## The pedagogical approach: pain first, theory second

This module opens with a **counter-example**. We deliberately plant a violation —
a probabilistic edge without provenance — into the SLGD and show what happens when
a compliance query runs against it. The query returns a confident-looking number.
Then we ask: how would a regulator know this number came from a probabilistic source?

The answer: they would not. That is the failure mode SHACL exists to prevent.

## Prerequisites

- Module 5 complete (SLGD populated with promoted entities and PROV-O provenance)
- `pyshacl` installed (pinned in `notebooks/shared/requirements.txt`)

## Deliverables

- `ontology/atlas-shapes.ttl` — SHACL shapes file with five shape categories
- A validation report (conformance or violations)
- `docs/model-risk-review.md` — plain-English explanation of each shape for MRM reviewers

## Architecture class for this module

**DETERMINISTIC.** SHACL validation is deterministic: the same data graph with the
same shapes graph always produces the same validation result.

## The Counter-Example: Why SHACL Exists

Before writing any shapes, let us see what happens **without** them.

The cell below plants a probabilistic edge — an LLM-suggested customer-to-product
affinity score — directly into a test graph without provenance, without a confidence
score, and without a model version. Then it runs a compliance-shaped query against it.

The query returns a confident-looking number. A regulator looking at that number
would have no way to know it came from a probabilistic source. That is the failure
mode. SHACL prevents it by making the boundary **structural** rather than **conventional**.

In [ ]:
# THE COUNTER-EXAMPLE: A probabilistic score without provenance
# This is what SHACL exists to prevent.

# Build a small test graph with a violation
g_bad = Graph()
g_bad.bind('atlas', ATLAS)
g_bad.bind('inst', INST)

# A customer with a score that has NO provenance, NO confidence, NO model version
cust = INST['customer-BAD-001']
score = INST['score-BAD-001']

g_bad.add((cust, RDF.type, ATLAS.Customer))
g_bad.add((score, RDF.type, ATLAS.Score))
g_bad.add((score, ATLAS.scoreValue, Literal('0.87', datatype=XSD.decimal)))
# MISSING: atlas:probabilistic, atlas:confidence, atlas:modelVersion, atlas:explainability
# MISSING: prov:wasGeneratedBy

# Now run a "compliance query" against it
query = """
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
SELECT ?score ?value WHERE {
    ?score a atlas:Score ;
           atlas:scoreValue ?value .
}
"""
results = list(g_bad.query(query))

print('THE PROBLEM: A compliance query returns a confident-looking number')
print('=' * 60)
print()
for row in results:
    print(f'  Score node: {row.score}')
    print(f'  Value:      {row.value}')
print()
print('A regulator sees: "Score 0.87"')
print('A regulator does NOT see:')
print('  - Was this deterministic or probabilistic? (unknown)')
print('  - What model produced it? (unknown)')
print('  - What confidence does the system have? (unknown)')
print('  - Can it be reproduced? (unknown)')
print()
print('This is the failure mode. The number looks authoritative but has no')
print('provenance. SHACL shapes prevent this by requiring structural attributes')
print('on every Score node BEFORE it can exist in the SLGD.')

## The Five SHACL Shapes

ATLAS defines five categories of shapes. Each enforces a different aspect of the
deterministic-vs-probabilistic boundary:

| # | Shape | What It Enforces | Why a Regulator Cares |
|---|-------|-----------------|----------------------|
| 1 | **Provenance shape** | Every entity in the SLGD must have `prov:wasDerivedFrom` or `atlas:promotedFrom` | Without provenance, you cannot trace a decision back to its source data |
| 2 | **Boundary shape** | Any property from a probabilistic source must be marked `atlas:probabilistic = true` and carry `atlas:confidence` in [0,1] | Without this marking, probabilistic outputs silently enter deterministic paths |
| 3 | **Compliance-input shape** | Any property tagged as a compliance input must NOT be probabilistic unless it also has `atlas:explainability = true` and `atlas:modelVersion` | Ensures ML outputs used in compliance decisions are explainable and version-pinned |
| 4 | **Routing-policy shape** | Any agent routing decision must reference a route from the closed SKOS scheme; arbitrary values fail | Prevents an LLM from inventing routes outside the enumerated set |
| 5 | **Cardinality shapes** | FIBO-aligned classes have correct cardinality (e.g., exactly one signal type per WealthSignal) | Catches routine modeling errors before they reach production |

The cell below writes these shapes to `ontology/atlas-shapes.ttl`.

In [ ]:
# Write the SHACL shapes file
shapes_content = '''@prefix sh:    <http://www.w3.org/ns/shacl#> .
@prefix atlas: <https://github.com/your-org/atlas/ontology#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .
@prefix prov:  <http://www.w3.org/ns/prov#> .
@prefix rdfs:  <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos:  <http://www.w3.org/2004/02/skos/core#> .

# ==========================================================================
# Shape 1: Provenance Shape
# Every promoted entity must have provenance (where it came from)
# ==========================================================================
atlas:ProvenanceShape
    a sh:NodeShape ;
    sh:targetClass atlas:Customer ;
    sh:property [
        sh:path atlas:promotedFrom ;
        sh:minCount 1 ;
        sh:message "Every promoted Customer must have atlas:promotedFrom linking to its LGD source. Without provenance, the audit trail is broken." ;
    ] .

# ==========================================================================
# Shape 2: Boundary Shape
# Scores must be marked as probabilistic with a confidence value
# ==========================================================================
atlas:BoundaryShape
    a sh:NodeShape ;
    sh:targetClass atlas:Score ;
    sh:property [
        sh:path atlas:probabilistic ;
        sh:minCount 1 ;
        sh:hasValue true ;
        sh:message "Every Score must have atlas:probabilistic = true. Scores are produced by ML models and must be explicitly marked as probabilistic." ;
    ] ;
    sh:property [
        sh:path atlas:confidence ;
        sh:minCount 1 ;
        sh:datatype xsd:decimal ;
        sh:minInclusive 0.0 ;
        sh:maxInclusive 1.0 ;
        sh:message "Every Score must have atlas:confidence between 0 and 1." ;
    ] .

# ==========================================================================
# Shape 3: Compliance-Input Shape
# Scores used as compliance inputs must be explainable and version-pinned
# ==========================================================================
atlas:ComplianceInputShape
    a sh:NodeShape ;
    sh:targetClass atlas:Score ;
    sh:property [
        sh:path atlas:explainability ;
        sh:minCount 1 ;
        sh:hasValue true ;
        sh:message "Scores must have atlas:explainability = true (SHAP attributions required for compliance use)." ;
    ] ;
    sh:property [
        sh:path atlas:modelVersion ;
        sh:minCount 1 ;
        sh:datatype xsd:string ;
        sh:message "Scores must have atlas:modelVersion identifying the model that produced them." ;
    ] .

# ==========================================================================
# Shape 4: Routing-Policy Shape
# Routing decisions must use values from the closed route scheme
# ==========================================================================
atlas:RoutingPolicyShape
    a sh:NodeShape ;
    sh:targetClass atlas:RoutingDecision ;
    sh:property [
        sh:path atlas:selectedRoute ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
        sh:in ( "ROUTE_ADVISOR_QUEUE"^^xsd:string "ROUTE_SUPPRESSION_LIST"^^xsd:string "ROUTE_ESCALATION"^^xsd:string ) ;
        sh:message "RoutingDecision must select exactly one route from the closed set: ROUTE_ADVISOR_QUEUE, ROUTE_SUPPRESSION_LIST, or ROUTE_ESCALATION. Arbitrary LLM-generated values are prohibited." ;
    ] .

# ==========================================================================
# Shape 5: Cardinality Shape (WealthSignal must have exactly one type)
# ==========================================================================
atlas:WealthSignalTypeShape
    a sh:NodeShape ;
    sh:targetClass atlas:WealthSignal ;
    sh:property [
        sh:path atlas:hasSignalType ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
        sh:message "Every WealthSignal must have exactly one signal type from the SKOS WealthSignalTypeScheme." ;
    ] .
'''

shapes_path = Path('../ontology/atlas-shapes.ttl')
shapes_path.write_text(shapes_content)
print(f'SHACL shapes written to {shapes_path}')
print(f'File size: {shapes_path.stat().st_size} bytes')
print()
print('Shapes defined:')
print('  1. ProvenanceShape     - targets atlas:Customer')
print('  2. BoundaryShape       - targets atlas:Score')
print('  3. ComplianceInputShape - targets atlas:Score')
print('  4. RoutingPolicyShape  - targets atlas:RoutingDecision')
print('  5. WealthSignalTypeShape - targets atlas:WealthSignal')

## Running the Validator

### First: validate the counter-example (should FAIL)

We run the shapes against the bad graph from the counter-example. The validator
should catch the missing provenance and boundary attributes.

In [ ]:
# Validate the counter-example graph against our shapes
shapes_graph = Graph()
shapes_graph.parse(str(shapes_path), format='turtle')

print('Validating counter-example (BAD graph) against SHACL shapes...')
print('=' * 60)

conforms, results_graph, results_text = pyshacl.validate(
    g_bad,
    shacl_graph=shapes_graph,
    inference='rdfs'
)

print(f'Conforms: {conforms}')
print()
if not conforms:
    print('VIOLATIONS FOUND (this is expected - the counter-example is deliberately bad):')
    print()
    print(results_text[:2000])
    print()
    print('The validator caught the missing attributes. This is exactly what SHACL')
    print('is for: making the boundary mechanical rather than conventional.')
else:
    print('[UNEXPECTED] Graph conforms - shapes may not be targeting correctly.')

### Second: validate correct data (should PASS)

Now we build a graph with properly attributed data — a Score with all required
fields — and confirm it passes validation.

In [ ]:
# Build a GOOD graph that should pass validation
g_good = Graph()
g_good.bind('atlas', ATLAS)
g_good.bind('inst', INST)
g_good.bind('prov', PROV)

# A properly attributed Score
score_good = INST['score-GOOD-001']
g_good.add((score_good, RDF.type, ATLAS.Score))
g_good.add((score_good, ATLAS.scoreValue, Literal('0.82', datatype=XSD.decimal)))
g_good.add((score_good, ATLAS.probabilistic, Literal(True, datatype=XSD.boolean)))
g_good.add((score_good, ATLAS.confidence, Literal('0.82', datatype=XSD.decimal)))
g_good.add((score_good, ATLAS.explainability, Literal(True, datatype=XSD.boolean)))
g_good.add((score_good, ATLAS.modelVersion, Literal('wealth-xgb-v1.0', datatype=XSD.string)))
g_good.add((score_good, PROV.wasGeneratedBy, INST['er-run-001']))

# A properly attributed Customer with provenance
cust_good = INST['customer-GOOD-001']
g_good.add((cust_good, RDF.type, ATLAS.Customer))
g_good.add((cust_good, ATLAS.promotedFrom, INST['lgd-customer-001']))

# A properly attributed RoutingDecision
route_good = INST['routing-GOOD-001']
g_good.add((route_good, RDF.type, ATLAS.RoutingDecision))
g_good.add((route_good, ATLAS.selectedRoute, Literal('ROUTE_ADVISOR_QUEUE', datatype=XSD.string)))

# A properly attributed WealthSignal
signal_good = INST['signal-GOOD-001']
g_good.add((signal_good, RDF.type, ATLAS.WealthSignal))
g_good.add((signal_good, ATLAS.hasSignalType, ATLAS.LargeDepositPattern))

print('Validating GOOD graph against SHACL shapes...')
print('=' * 60)

conforms, results_graph, results_text = pyshacl.validate(
    g_good,
    shacl_graph=shapes_graph,
    inference='rdfs'
)

print(f'Conforms: {conforms}')
print()
if conforms:
    print('[PASS] The properly attributed graph passes all five shapes.')
    print()
    print('What made it pass:')
    print('  - Score has: probabilistic=true, confidence=0.82, explainability=true, modelVersion')
    print('  - Customer has: promotedFrom (provenance to LGD source)')
    print('  - RoutingDecision has: selectedRoute from the closed set')
    print('  - WealthSignal has: exactly one hasSignalType')
else:
    print('[UNEXPECTED FAIL] Good graph has violations:')
    print(results_text[:1000])

## Module 6 Validation Gate

The gate checks:
1. `atlas-shapes.ttl` parses as valid SHACL
2. The counter-example (bad graph) fails validation (shapes catch violations)
3. The good graph passes validation (no false positives)
4. All five shape categories are present in the shapes file
5. The shapes file is non-empty and well-formed

In [ ]:
print('=' * 60)
print('MODULE 6 VALIDATION GATE')
print('=' * 60)
print()

gate_pass = True

# Gate 1: Shapes file parses
try:
    g_shapes_check = Graph()
    g_shapes_check.parse(str(shapes_path), format='turtle')
    print(f'[PASS] Gate 1 - atlas-shapes.ttl parses ({len(g_shapes_check)} triples)')
except Exception as e:
    print(f'[FAIL] Gate 1 - Parse error: {e}')
    gate_pass = False

# Gate 2: Bad graph fails validation
conforms_bad, _, _ = pyshacl.validate(g_bad, shacl_graph=shapes_graph, inference='rdfs')
if not conforms_bad:
    print(f'[PASS] Gate 2 - Counter-example correctly fails validation')
else:
    print(f'[FAIL] Gate 2 - Counter-example should fail but conforms')
    gate_pass = False

# Gate 3: Good graph passes validation
conforms_good, _, _ = pyshacl.validate(g_good, shacl_graph=shapes_graph, inference='rdfs')
if conforms_good:
    print(f'[PASS] Gate 3 - Good graph correctly passes validation')
else:
    print(f'[FAIL] Gate 3 - Good graph should pass but has violations')
    gate_pass = False

# Gate 4: All five shapes present
shapes_text = shapes_path.read_text()
required_shapes = ['ProvenanceShape', 'BoundaryShape', 'ComplianceInputShape', 'RoutingPolicyShape', 'WealthSignalTypeShape']
missing = [s for s in required_shapes if s not in shapes_text]
if not missing:
    print(f'[PASS] Gate 4 - All 5 shape categories present')
else:
    print(f'[FAIL] Gate 4 - Missing shapes: {missing}')
    gate_pass = False

# Gate 5: File is non-trivial
if len(shapes_text) > 1000:
    print(f'[PASS] Gate 5 - Shapes file is {len(shapes_text)} bytes (non-trivial)')
else:
    print(f'[FAIL] Gate 5 - Shapes file too small ({len(shapes_text)} bytes)')
    gate_pass = False

print()
if gate_pass:
    print('MODULE 6 VALIDATION: PASS')
    print('You may proceed to Module 7.')
else:
    print('MODULE 6 VALIDATION: FAIL')
    raise AssertionError('Module 6 validation gate failed.')

## Extending This to Your Data

### Two additional shapes your program will likely need

**Data-residency shape**: Any entity whose source jurisdiction is restricted must
not be replicated to a graph in a different jurisdiction. Write a shape that checks
`atlas:sourceJurisdiction` against `atlas:graphJurisdiction` and flags mismatches.

**Retention shape**: Any entity whose source carries a retention policy must have
an `atlas:expiresAt` date. The SLGD's purge job respects this date. Without the
shape, data without expiry dates accumulates indefinitely.

### The most common SHACL gotcha

**Closed-shape semantics.** A closed shape (`sh:closed true`) rejects any predicate
not explicitly enumerated in the shape. This surprises teams when they add a new
property to the ontology and the validator suddenly fails on every entity.

The fix: do not use `sh:closed true` on shapes that target classes you expect to
extend. Use it only on shapes where the set of allowed predicates is genuinely fixed
(like RoutingDecision, where the route set is closed by design).

### Writing the model-risk-review.md document

The plain-English document (`docs/model-risk-review.md`) explains each shape in
language an MRM reviewer can read without knowing Turtle syntax. For each shape:
1. What it enforces (one sentence)
2. Why a regulator cares (one sentence)
3. What a violation looks like (one example)
4. How to fix a violation (one instruction)

This document is itself a deliverable — model risk management cannot review what
it cannot read.

## What Changed

| Artifact | Location | Description |
|----------|----------|-------------|
| SHACL shapes | `ontology/atlas-shapes.ttl` | Five shape categories enforcing the deterministic-vs-probabilistic boundary |
| Validation demonstrated | This notebook | Counter-example fails, good data passes |
| Model risk review | `docs/model-risk-review.md` | Plain-English shape explanations for MRM reviewers |

**Key architectural point established:**

The boundary is no longer a slogan or a convention. It is a property the system has,
observable from outside. A reviewer can run `pyshacl` against the SLGD and produce
a report showing exactly where probabilistic outputs entered the system and exactly
which constraints prevent them from corrupting compliance-bound paths.

**What Module 7 builds on this:**

Module 7 introduces Amazon Bedrock at the edges — translating natural language to
SPARQL queries. Every LLM-generated query passes through a SHACL pre-check before
execution: if the query would write probabilistic-opaque data to the SLGD, the
shape catches it before the write reaches Neptune.